# Simulation result explorer

Load one simulation result store, select its last shared step, and explore the terminal observer values against the sampled parameters. Edit the result path and observer metadata in the configuration cell; the final cell displays the widget.

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import scienceplots  # noqa: F401
from IPython.display import display
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.ticker import MaxNLocator
from matplotlib_inline.backend_inline import set_matplotlib_formats

%matplotlib inline
plt.style.use("science")
set_matplotlib_formats("svg")
extra = 3
plt.rcParams.update({
    "font.size": 14+extra,
    "axes.titlesize": 17+extra,
    "axes.labelsize": 15+extra,
    "xtick.labelsize": 12+extra,
    "ytick.labelsize": 12+extra,
    "legend.fontsize": 12+extra,
    "legend.title_fontsize": 13+extra,
    "figure.titlesize": 19+extra,
})

project_root = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "pyproject.toml").is_file() and (path / "tasks").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from the network-creation repository.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from network_creation.plotting.plot_pair import plot_pair
from network_creation.plotting.reference_values import get_observer_reference_style
from tasks.policies import load_policy_results

In [ ]:
CONFIG = {
    "result_store": (
        project_root
        / "artifacts"
        / "simulation_runs"
        / "run-20260713T235130.zarr"
    ),
    "normalize_time": True,
    "export": {
        "output_root": project_root / "artifacts" / "figures",
        "dpi": 300,
    },
    "observables": {
        "centered_mean_welfare.mean": {
            "label": "Mean welfare at the last step",
            "limits": (0.0, 1.0),
        },
        "centered_mean_welfare.std": {
            "label": "Welfare SD at the last step",
            "limits": (0.0, None),
        },
    },
    "plot": {
        "method": "gaussian",
        "cmap": "jet",
        "resolution": 50,
        "point_size": 4,
        "pair_atol": 1e-5,
        "figure_side": 4,
        "subplot_spacing": 0.06,
        "subplot_margins": {
            "left": 0.125,
            "right": 0.895,
            "bottom": 0.11,
            "top": 0.88,
        },
        "colorbar_bounds": (0.35, 0.0, 0.08, 1.0),
        "single_colorbar_bounds": (1.08, 0.0, 0.05, 1.0),
        "under_range_fraction": 0.16,
        "parameter_labels": {
            "visibility_distance": "Visibility radius",
            "exploration": "Decision temperature",
            "outside_optimism": "Unseen-spillover weight",
        },
        "reference_labels": {
            "star": "Star",
            "random": "Max ER",
            "choi2020": "Choi (2020)",
            "empty": "Empty",
        },
    },
}

In [ ]:
results = load_policy_results(
    CONFIG["result_store"],
    stats=tuple(CONFIG["observables"]),
    normalize_time=CONFIG["normalize_time"],
)
last_step = results["x"].isel(step=-1, drop=True).rename("last_step")

composition_dims = tuple(
    dim
    for dim in results["x"].dims
    if dim not in {"n_agents", "parameter_row", "step", "observable"}
)
if len(composition_dims) != 1:
    raise ValueError(
        f"Expected one policy-composition dimension, found {composition_dims}."
    )
composition_dim = composition_dims[0]

loaded_path = Path(results.attrs["result_store_path"])
print(f"Loaded {loaded_path.relative_to(project_root)}")
print(f"Has failures: {results.attrs['has_failures']}")
print(f"Last shared step: {float(results.coords['step'][-1]):g}")
print(f"Time axis: {results.attrs.get('time_axis', 'raw_simulator_steps')}")

In [ ]:
def resolve_norm(observable, values, references, suggested_norm):
    finite = np.asarray(values, dtype=float).ravel()
    finite = finite[np.isfinite(finite)]
    if references:
        finite = np.concatenate([finite, np.asarray(list(references.values()))])
    if not finite.size:
        raise ValueError(f"No finite values are available for {observable!r}.")

    configured_min, configured_max = CONFIG["observables"][observable]["limits"]
    vmin = float(finite.min()) if configured_min is None else float(configured_min)
    vmax = float(finite.max()) if configured_max is None else float(configured_max)
    if isinstance(suggested_norm, TwoSlopeNorm) and vmin < suggested_norm.vcenter < vmax:
        return TwoSlopeNorm(vmin=vmin, vcenter=suggested_norm.vcenter, vmax=vmax)
    return Normalize(vmin=vmin, vmax=vmax)


def draw_color_scale(axis, norm, cmap, references, label, under_fraction):
    gradient = np.linspace(0.0, 1.0, 512)[:, None]
    axis.set_facecolor(cmap(0.0))
    axis.imshow(
        gradient,
        cmap=cmap,
        vmin=0.0,
        vmax=1.0,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        extent=(0.0, 1.0, 0.0, 1.0),
    )
    axis.set_xlim(0.0, 1.0)
    axis.set_ylim(-under_fraction, 1.0)
    axis.set_xticks([])

    if references:
        items = sorted(references.items(), key=lambda item: item[1])
        tick_values = [value for _, value in items]
        tick_labels = [
            f"{CONFIG['plot']['reference_labels'].get(name, name)} ({value:.2f})"
            for name, value in items
        ]
    else:
        tick_values = MaxNLocator(nbins=6).tick_values(norm.vmin, norm.vmax)
        tick_values = [
            value for value in tick_values if norm.vmin <= value <= norm.vmax
        ]
        tick_labels = [f"{value:.2f}" for value in tick_values]

    axis.set_yticks(
        [float(norm(value)) for value in tick_values],
        labels=tick_labels,
    )
    axis.minorticks_off()
    axis.yaxis.tick_right()
    axis.yaxis.set_label_position("left")
    axis.set_ylabel(label)
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)


def plot_last_step(observable, n_agents, composition):
    selector = {
        "observable": observable,
        "n_agents": n_agents,
        composition_dim: composition,
    }
    values = last_step.sel(selector)
    theta = results["theta"].sel(
        n_agents=n_agents,
        **{composition_dim: composition},
    )
    valid = values.notnull() & theta.notnull().all("theta_parameter")
    values = values.where(valid, drop=True)
    theta = theta.where(valid, drop=True)
    if not values.sizes.get("parameter_row", 0):
        raise ValueError("No finite final values match this selection.")

    plot_config = CONFIG["plot"]
    varying_parameters = int(
        (theta.var("parameter_row") > plot_config["pair_atol"]).sum().item()
    )
    n_axes = varying_parameters - 1
    if n_axes < 1:
        raise ValueError("At least two sampled parameters must vary for a pair plot.")

    theta_names = [str(name) for name in theta.coords["theta_parameter"].values]
    parameter_labels = plot_config["parameter_labels"]
    labels = [parameter_labels.get(name, name) for name in theta_names]

    references = {}
    suggested_norm = None
    if observable.split(".")[1:2] == ["mean"]:
        references, suggested_norm = get_observer_reference_style(
            observable,
            values.to_numpy(),
            n_agents=int(n_agents),
            theta_labels=theta_names,
            theta_values=theta.to_numpy(),
        )

    shared_values = last_step.sel(
        observable=observable,
        **{composition_dim: composition},
    ).to_numpy()
    norm = resolve_norm(observable, shared_values, references, suggested_norm)
    base_cmap = plt.get_cmap(plot_config["cmap"])
    cmap = base_cmap.with_extremes(under=base_cmap(0.0))
    has_under_range = bool(
        norm.vmin is not None
        and np.any(np.isfinite(shared_values) & (shared_values < norm.vmin))
    )

    with plt.ioff():
        fig, axes = plt.subplots(
            n_axes,
            n_axes,
            squeeze=False,
            figsize=(plot_config["figure_side"] * n_axes,) * 2,
            gridspec_kw={
                "wspace": plot_config["subplot_spacing"],
                "hspace": plot_config["subplot_spacing"],
            },
        )
        fig.subplots_adjust(**plot_config["subplot_margins"])
    plot_pair(
        X=theta.to_numpy(),
        y=values.to_numpy(),
        labels=labels,
        ordered_discrete_keys=[
            parameter_labels.get("visibility_distance", "visibility_distance")
        ]
        if "visibility_distance" in theta_names
        else [],
        atol=plot_config["pair_atol"],
        cmap=cmap,
        method=plot_config["method"],
        resolution=plot_config["resolution"],
        point_size=plot_config["point_size"],
        norm=norm,
        fig=fig,
        axes=axes,
    )

    if n_axes == 1:
        colorbar_axis = axes[0, 0].inset_axes(
            plot_config["single_colorbar_bounds"]
        )
    else:
        colorbar_slot = axes[0, -1]
        colorbar_slot.clear()
        colorbar_slot.axis("off")
        colorbar_axis = colorbar_slot.inset_axes(plot_config["colorbar_bounds"])
    draw_color_scale(
        colorbar_axis,
        norm,
        cmap,
        references,
        CONFIG["observables"][observable]["label"],
        plot_config["under_range_fraction"] if has_under_range else 0.0,
    )
    fig.align_ylabels(axes[:, 0])

    composition_label = composition_dim.replace("_", " ")
    """
    fig.suptitle(
        f"{CONFIG['observables'][observable]['label']}\n"
        f"n_agents={int(n_agents)}, {composition_label}={composition:g}"
    )
    """
    return fig

In [ ]:
control_style = {"description_width": "initial"}
controls = {
    "observable": widgets.Dropdown(
        description="Observer",
        options=[
            (metadata["label"], observable)
            for observable, metadata in CONFIG["observables"].items()
        ],
        style=control_style,
        layout=widgets.Layout(width="320px"),
    ),
    "n_agents": widgets.Dropdown(
        description="n agents",
        options=[int(value) for value in results.coords["n_agents"].values],
        style=control_style,
    ),
    "composition": widgets.Dropdown(
        description=composition_dim.replace("_", " "),
        options=[
            (f"{value:g}", value)
            for value in results.coords[composition_dim].values.tolist()
        ],
        style=control_style,
    ),
}

plot_controls = widgets.HBox(
    list(controls.values()),
    layout=widgets.Layout(
        gap="12px",
        align_items="flex-start",
        flex_flow="row wrap",
    ),
)


def build_plot():
    figure = plot_last_step(
        controls["observable"].value,
        controls["n_agents"].value,
        controls["composition"].value,
    )
    plt.close(figure)
    return figure


def update_plot(_):
    try:
        plot_display.update(build_plot())
    except ValueError as error:
        plot_display.update(str(error))


display(plot_controls)
plot_display = display(build_plot(), display_id=True)
for control in controls.values():
    control.observe(update_plot, names="value")

In [ ]:
run_id = Path(CONFIG["result_store"]).stem
figure_dir = CONFIG["export"]["output_root"] / run_id
figure_dir.mkdir(parents=True, exist_ok=True)

pdf_paths = []
for observable in CONFIG["observables"]:
    for n_agents in results.coords["n_agents"].values:
        for composition in results.coords[composition_dim].values:
            figure = plot_last_step(
                observable, int(n_agents), composition.item()
            )
            filename = (
                f"{observable}__n_agents-{int(n_agents)}__"
                f"{composition_dim}-{composition:g}.pdf"
            )
            pdf_path = figure_dir / filename
            figure.savefig(
                pdf_path,
                format="pdf",
                dpi=CONFIG["export"]["dpi"],
                bbox_inches="tight",
            )
            plt.close(figure)
            pdf_paths.append(pdf_path)

print(
    f"Saved {len(pdf_paths)} PDFs to "
    f"{figure_dir.relative_to(project_root)}"
)